# Example Dataset
For the remainder of this lesson, we will walk slowly through the process of building a multilevel/mixed-effects model for a single example data structure. This will be broken down into the individual steps described previously. Each step will contain a reasonable amount of exposition so that you can see how the generic steps can be applied to this specific example. This will include all the decision making and logic behind our final choices. The hope is that, by the end, the process itself will be clearer. In the associated workshop, there will be multiple examples to choose from, so you can see the same steps and logic applied to other types of data structure.

## The Rat Pup Data
... Of course, in the realm of psychology you may not ever find yourself working with baby rats. However, it is useful to see many different examples from different fields so you get a sense of how these methods can be applied *generically*. Otherwise, it is quite easy to get stuck with only conceptualising these approaches in terms of `subjects` and nothing else.

### Data Description


### Downloading and Wrangling the Data
... The data are available from [here](https://websites.umich.edu/~bwest/chapter3.html) and need to be downloaded into the current working directory before being loaded. In these examples, we have put the data in a separate `data` folder, so just be aware of this if you are copying and pasting code.

In [1]:
ratpup <- read.table('data/rat_pup.dat', header=TRUE)
ratpup[1:24,]

   pup_id weight    sex litter litsize treatment
1       1   6.60   Male      1      12   Control
2       2   7.40   Male      1      12   Control
3       3   7.15   Male      1      12   Control
4       4   7.24   Male      1      12   Control
5       5   7.10   Male      1      12   Control
6       6   6.04   Male      1      12   Control
7       7   6.98   Male      1      12   Control
8       8   7.05   Male      1      12   Control
9       9   6.95 Female      1      12   Control
10     10   6.29 Female      1      12   Control
11     11   6.77 Female      1      12   Control
12     12   6.57 Female      1      12   Control
13     13   6.37   Male      2      14   Control
14     14   6.37   Male      2      14   Control
15     15   6.90   Male      2      14   Control
16     16   6.34   Male      2      14   Control
17     17   6.50   Male      2      14   Control
18     18   6.10   Male      2      14   Control
19     19   6.44   Male      2      14   Control
20     20   6.94   M

These data are already in *long-format*, so all we need to do is make sure that all categorical variables are correctly defined as a `factor` using

In [2]:
ratpup$sex       <- as.factor(ratpup$sex)
ratpup$litter    <- as.factor(ratpup$litter)
ratpup$treatment <- as.factor(ratpup$treatment)

## Identifying the Structure
As a first step, we need to determine what *structure* we are dealing with, as this provides us with the first clues around how to start modelling these data.

### 1. Identify the Units of Analysis
To begin, we need to identify our *unit of analysis*. This is the element of the data that we are trying to make inferences on. So, in this example, our units are *individual rat pups*. Our primary question is about how the treatments affect the weight of the rat pups. 

### 2. Identify the Rows of the Data
In the second step, we need to determine what each row of the data represents. As we can see above, each row corresponds to only a single unit measured *once*. So this is *not* repeated measures or longitudinal data, but it *could* be *clustered* data. 

### 3. Identify Clusters
In the final step, we need to determine if there are any clusters. There *may* or *may not* be, and there also may be several nested clusters.

If there are, we use a multilevel/LME approach. If there are not, the data are *cross-sectional* and we go back to normal linear regression. So, we look and see whether there are any variables that are *constant* across multiple units. In this example, there are several: `sex`, `litter`, `litsize` and `treatment`. The important question is whether any of these represent a *shared latent influence* that could affect the correlation between pups? If we think back to our *sticky notes* vs *rubber bands* metaphor, what elements of the data tie all those shared units together and which are simply labels that we give those units? 

In this example, `sex`, `litsize` and `treatment` are all *sticky notes*. They are *labels* given to each individual unit. They describe a *characteristic* of the pup or a *characteristic* of their environment, but they do not tie them together. By comparison, `litter` itself is a *rubber band*. All pups from the same litter are tied together by the biological reality of sharing a mother, genes, siblings and rearing environment. So, our conclusion is that `litter` *is* a clustering variable. All pups from the same `litter` are therefore expected to be correlated to some degree.

````{admonition} More on Litter Size
:class: tip, dropdown
You may find `litsize` a confusing inclusion because it *refers* to litter and thus sounds like it should relate to clustering. However, this is a *description* rather than a structure. The way to think about it is if, hypothetically, litter size was a *rubber band* it would mean that all pups from litters of the same size were connected, irrespective of their mother, rearing environment or siblings. So, it is simply the total number of pups in the litter that matters biologically, rather than anything else. Thus all pups from a litter of size 8 would be connected together. Their mother, genetics and other factors *do not matter*. Consider for yourself whether this sounds plausible and thus whether you think `litsize` should be treated as a *rubber band* or a *sticky note*. 
````